 ## 0. Описание лабораторной работы

Вам предоставлен набор данных, который содержит информацию о фильмах:

**Таблица mkrf_movies** содержит информацию из реестра прокатных удостоверений:
- **title** — название фильма
- **puNumber** — номер прокатного удостоверения
- **show_start_date** — дата премьеры фильма
- **type** — тип фильма
- **film_studio** — студия-производитель
- **production_country** — страна-производитель
- **director** — режиссёр
- **producer** — продюсер
- **age_restriction** — возрастная категория
- **refundable_support** — объём возвратных средств государственной поддержки
- **nonrefundable_support** — объём невозвратных средств государственной поддержки
- **financing_source** — источник государственного финансирования
- **budget** — общий бюджет фильма
- **ratings** — рейтинг фильма на КиноПоиске
- **genres** — жанр фильма

**Таблица mkrf_shows** содержит сведения о показах фильмов в российских кинотеатрах:
- **puNumber** — номер прокатного удостоверения
- **box_office** — сборы в рублях

*Примечание: у одного фильма может быть несколько прокатных удостоверений*

**Задача:** предсказать рейтинг фильма.

**Задание:**

**1. Анализ и предобработка**
- Проанализировать данные (EDA)
- Предобработать данные
- Скалировать/нормализовать данные
- Подготовить данные для обучения моделей

**2. Построение baseline-модели нейронной сети**
- Создайте класс для архитектуры нейронной сети (используйте фреймворк на выбор — Keras, PyTorch, TensorFlow)
- Самостоятельно выберите:
  - Количество скрытых слоёв
  - Количество нейронов в каждом слое
  - Функции активации для скрытых и выходного слоёв
- Реализуйте обучение модели: напишите функцию обучения и выберите метрику для оценки

**3. Доработка модели**
- Реализуйте подбор параметров (grid search) для следующих гиперпараметров:
  - Функция активации
  - Dropout (наличие и значение)
  - Batch Normalization (наличие)
  - Размер батча
- Архитектуру (количество слоёв и нейронов) оставьте как в Baseline
- Визуализируйте зависимость метрики RMSE от значений подбираемых параметров
- Сравните результаты разных комбинаций параметров, выберите лучшую модель по метрике RMSE
- Сделайте вывод по работе: проанализируйте, какие подходы оказались наиболее эффективны для поставленной задачи

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import plotly.io as pio
import plotly.express as px
pio.templates.default = 'plotly_white'
from plotly.subplots import make_subplots
import plotly.graph_objects as go

import polars as pl
import numpy as np
import random
import sklearn
import os

seed = 42
random.seed(seed)
np.random.seed(seed)
sklearn.random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)

## 1. Анализ и предобработка


In [ ]:
movies_data_url = 'https://github.com/Lopa10ko/itmo-ml-2025/raw/main/lab-6/mkrf_movies.csv'
movies_pl = pl.read_csv(movies_data_url,
                        schema_overrides={'ratings': pl.String},
                        null_values=['нет'])

string_columns = movies_pl.select(pl.col(pl.Utf8)).columns

# уберем сразу окружающие строковые значения пробелы
movies_pl = movies_pl.with_columns([
    pl.col(col).str.strip_chars().alias(col)
    for col in string_columns
])

shows_data_url = 'https://github.com/Lopa10ko/itmo-ml-2025/raw/main/lab-6/mkrf_shows.csv'
shows_pl = pl.read_csv(shows_data_url)

display(movies_pl.head(3))
print(f'Размер датасета mkrf_movies: {movies_pl.shape}\n')

display(shows_pl.head(3))
print(f'Размер датасета mkrf_shows: {shows_pl.shape}\n')

title,puNumber,show_start_date,type,film_studio,production_country,director,producer,age_restriction,refundable_support,nonrefundable_support,budget,financing_source,ratings,genres
str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str
"""Открытый простор""",221048915,"""2015-11-27T12:00:00.000Z""","""Художественный""","""Тачстоун Пикчерз, Кобальт Пикч…","""США""","""Кевин Костнер""","""Дэвид Валдес, Кевин Костнер, Д…","""«18+» - запрещено для детей""",null,null,null,null,"""7.2""","""боевик,драма,мелодрама"""
"""Особо важное задание""",111013716,"""2016-09-13T12:00:00.000Z""","""Художественный""","""Киностудия ""Мосфильм""""","""СССР""","""Е.Матвеев""",null,"""«6+» - для детей старше 6 лет""",null,null,null,null,"""6.6""","""драма,военный"""
"""Особо опасен""",221038416,"""2016-10-10T12:00:00.000Z""","""Художественный""","""Юниверсал Пикчерз, Кикстарт Пр…","""США""","""Тимур Бекмамбетов""","""Джим Лемли, Джейсон Нетер, Мар…","""«18+» - запрещено для детей""",null,null,null,null,"""6.8""","""фантастика,боевик,триллер"""


Размер датасета mkrf_movies: (7486, 15)



puNumber,box_office
i64,f64
111000113,2450.0
111000115,61040.0
111000116,1.5303e8


Размер датасета mkrf_shows: (3158, 2)



На этапе загрузки данных обнаружена несогласованность форматов в рамках одного атрибута (в частности, рейтинга).

Для обеспечения целостности данных признак рейтинга был импортирован как строковая переменная, что позволило унифицировать обработку различных систем оценки: 10-балльной шкалы (например, 8.2, как на Кинопоиске), процентных соотношений (например, рейтинг в 97%) и текстовых маркеров отсутствия рейтинга (например, рейтинг "нет").

Далее еще вернемся к обработке значений этого признака

Агрегируем данные о показах по puNumber (суммируем сборы) и объединим с основной таблицей о фильмах по прокатным удостоверениям

In [ ]:
shows_aggregated = shows_pl.group_by('puNumber').agg([
    pl.sum('box_office').alias('total_box_office'),
    pl.count().alias('show_count')
])

df_pl = movies_pl.join(shows_aggregated, on='puNumber', how='left')

display(df_pl.head(10))
print(f'Размер общего датасета: {df_pl.shape}\n')

title,puNumber,show_start_date,type,film_studio,production_country,director,producer,age_restriction,refundable_support,nonrefundable_support,budget,financing_source,ratings,genres,total_box_office,show_count
str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,u32
"""Открытый простор""",221048915,"""2015-11-27T12:00:00.000Z""","""Художественный""","""Тачстоун Пикчерз, Кобальт Пикч…","""США""","""Кевин Костнер""","""Дэвид Валдес, Кевин Костнер, Д…","""«18+» - запрещено для детей""",null,null,null,null,"""7.2""","""боевик,драма,мелодрама""",null,null
"""Особо важное задание""",111013716,"""2016-09-13T12:00:00.000Z""","""Художественный""","""Киностудия ""Мосфильм""""","""СССР""","""Е.Матвеев""",null,"""«6+» - для детей старше 6 лет""",null,null,null,null,"""6.6""","""драма,военный""",null,null
"""Особо опасен""",221038416,"""2016-10-10T12:00:00.000Z""","""Художественный""","""Юниверсал Пикчерз, Кикстарт Пр…","""США""","""Тимур Бекмамбетов""","""Джим Лемли, Джейсон Нетер, Мар…","""«18+» - запрещено для детей""",null,null,null,null,"""6.8""","""фантастика,боевик,триллер""",null,null
"""Особо опасен""",221026916,"""2016-06-10T12:00:00.000Z""","""Художественный""","""Юниверсал Пикчерз, Кикстарт Пр…","""США""","""Тимур Бекмамбетов""","""Джим Лемли, Джейсон Нетер, Мар…","""«18+» - запрещено для детей""",null,null,null,null,"""6.8""","""фантастика,боевик,триллер""",null,null
"""Особо опасен""",221030815,"""2015-07-29T12:00:00.000Z""","""Художественный""","""Юниверсал Пикчерз, Кикстарт Пр…","""США""","""Тимур Бекмамбетов""","""Джим Лемли, Джейсон Нетер, Мар…","""«18+» - запрещено для детей""",null,null,null,null,"""6.8""","""фантастика,боевик,триллер""",null,null
"""Остановился поезд""",111013816,"""2016-09-13T12:00:00.000Z""","""Художественный""","""Киностудия ""Мосфильм""""","""СССР""","""В.Абдрашитов""",null,"""«6+» - для детей старше 6 лет""",null,null,null,null,"""7.7""","""драма""",null,null
"""Любовь и голуби""",111007013,"""2013-10-18T12:00:00.000Z""","""Художественный""","""Киностудия ""Мосфильм""""","""СССР""","""В.Меньшов""",null,"""«12+» - для детей старше 12 ле…",null,null,null,null,"""8.3""","""мелодрама,комедия""",2700.0,1
"""Любовь и сигареты""",221074614,"""2014-12-29T12:00:00.000Z""","""Художественный""","""Юнайтед Артистс, Грин Стрит Фи…","""США""","""Джон Туртурро""","""Джон Пенотти, Джон Туртурро""","""«18+» - запрещено для детей""",null,null,null,null,"""6.6""","""мюзикл,мелодрама,комедия""",null,null
"""Отпетые мошенники.""",121011416,"""2016-05-05T12:00:00.000Z""","""Художественный""","""Пульсар Продюксьон, ТФ1 Фильм""","""Франция""","""Эрик Беснард""","""Патрис Леду""","""«18+» - запрещено для детей""",null,null,null,null,"""8.0""","""комедия,криминал""",null,null


Размер общего датасета: (7486, 17)



Но заметим также важное уточнение из условия лабораторной о том, что количество уникальных картин (по названию) не соотвествует количеству уникальных прокатных удостоверений

In [ ]:
print(f'n_unique puNumber: {df_pl["puNumber"].n_unique()}')
print(f'n_unique title: {df_pl["title"].n_unique()}')

n_unique puNumber: 7484
n_unique title: 6772


Получается, что в нашем датасете сейчас находятся дубликаты по названию фильма. Агрегировать остается лишь рейтинг и total_box_office, но опять же, сделаем это чуть позже (после заполнения вещественных признаков)

In [ ]:
key_features = ['title', 'type', 'production_country']

duplicate_combinations = df_pl.select(key_features).filter(
    pl.struct(key_features).is_duplicated()
).unique()

print(f"Уникальные комбинации с дубликатами: {duplicate_combinations.height}")

for i, combo in enumerate(duplicate_combinations.head(5).iter_rows(named=True)):
    print(f"\n--- Дубликаты группы {i + 1} ---")
    group_data = df_pl.filter(
        (pl.col('title') == combo['title']) &
        (pl.col('type') == combo['type']) &
        (pl.col('production_country') == combo['production_country'])
    )
    display(group_data)

Уникальные комбинации с дубликатами: 535

--- Дубликаты группы 1 ---


title,puNumber,show_start_date,type,film_studio,production_country,director,producer,age_restriction,refundable_support,nonrefundable_support,budget,financing_source,ratings,genres,total_box_office,show_count
str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,u32
"""7 дней и ночей с Мэрилин""",121008016,"""2016-04-05T12:00:00.000Z""","""Художественный""","""БиБиСи Филмз, Вайнштейн Компан…","""Великобритания-США""","""Саймон Кертис""","""Дэвид Парфитт""","""«18+» - запрещено для детей""",null,null,null,null,"""6.9""","""драма,биография""",null,null
"""7 дней и ночей с Мэрилин""",121024711,"""2011-12-27T12:00:00.000Z""","""Художественный""","""БиБиСи Филмз, Вайнштейн Компан…","""Великобритания-США""","""Саймон Кертис""","""Дэвид Парфитт""","""«18+» - запрещено для детей""",null,null,null,null,"""6.9""","""драма,биография""",null,null



--- Дубликаты группы 2 ---


title,puNumber,show_start_date,type,film_studio,production_country,director,producer,age_restriction,refundable_support,nonrefundable_support,budget,financing_source,ratings,genres,total_box_office,show_count
str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,u32
"""Катись!""",221018710,"""2010-02-19T12:00:00.000Z""","""Художественный""","""Мэндейт Пикчерз, Винсент Пикче…","""США""","""Дрю Бэрримор""","""Николь Браун, Дрю Бэрримор, Пи…","""«16+» - для детей старше 16 ле…",null,null,null,null,"""6.8""","""драма,спорт""",null,null
"""Катись!""",121002510,"""2010-02-11T12:00:00.000Z""","""Художественный""","""Мэндейт Пикчерз, Винсент Пикче…","""США""","""Дрю Бэрримор""","""Николь Браун, Дрю Бэрримор, Пи…","""«16+» - для детей старше 16 ле…",null,null,null,null,"""6.8""","""драма,спорт""",null,null



--- Дубликаты группы 3 ---


title,puNumber,show_start_date,type,film_studio,production_country,director,producer,age_restriction,refundable_support,nonrefundable_support,budget,financing_source,ratings,genres,total_box_office,show_count
str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,u32
"""Охотники за головами""",221016216,"""2016-03-02T12:00:00.000Z""","""Художественный""","""Фрилэнд, Йеллоу Берд Филмз, Но…","""Норвегия - Швеция - Дания - Ге…","""Мортен Тильдум""","""Марианн Грэй, Асле Ватн""","""«18+» - запрещено для детей""",null,null,null,null,"""7.5""","""боевик,триллер,криминал""",null,null
"""Охотники за головами""",121023711,"""2011-12-14T12:00:00.000Z""","""Художественный""","""Фрилэнд, Йеллоу Берд Филмз, Но…","""Норвегия - Швеция - Дания - Ге…","""Мортен Тильдум""","""Марианн Грэй, Асле Ватн""","""«18+» - запрещено для детей""",null,null,null,null,"""7.5""","""боевик,триллер,криминал""",null,null



--- Дубликаты группы 4 ---


title,puNumber,show_start_date,type,film_studio,production_country,director,producer,age_restriction,refundable_support,nonrefundable_support,budget,financing_source,ratings,genres,total_box_office,show_count
str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,u32
"""Как выйти замуж за миллиардера""",121009416,"""2016-04-21T12:00:00.000Z""","""Художественный""","""Кросс Дэй Продакшн, Калейдоско…","""Германия - Великобритания - Ав…","""Фил Трэйл""","""Вольфганг Бер, Дитмар Гюнче, П…","""«18+» - запрещено для детей""",null,null,null,null,"""6.9""","""мелодрама,комедия,спорт""",null,null
"""Как выйти замуж за миллиардера""",121004111,"""2011-03-25T12:00:00.000Z""","""Художественный""","""Кросс Дэй Продакшн, Калейдоско…","""Германия - Великобритания - Ав…","""Фил Трэйл""","""Вольфганг Бер, Дитмар Гюнче, П…","""«18+» - запрещено для детей""",null,null,null,null,"""6.9""","""мелодрама,комедия,спорт""",null,null



--- Дубликаты группы 5 ---


title,puNumber,show_start_date,type,film_studio,production_country,director,producer,age_restriction,refundable_support,nonrefundable_support,budget,financing_source,ratings,genres,total_box_office,show_count
str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,u32
"""Золушка: Полный вперед!""",224001416,"""2016-04-07T12:00:00.000Z""","""Анимационный""","""ЮФилм, Эрольд энд Фэмили, МК2 …","""Франция""","""Паскаль Эрольд""","""Паскаль Эрольд""","""«12+» - для детей старше 12 ле…",null,null,null,null,"""4.2""","""мультфильм,фэнтези,семейный""",null,null
"""Золушка: Полный вперед!""",124001612,"""2012-07-06T12:00:00.000Z""","""Анимационный""","""ЮФилм, Эрольд энд Фэмили, МК2 …","""Франция""","""Паскаль Эрольд""","""Паскаль Эрольд""","""«12+» - для детей старше 12 ле…",null,null,null,null,"""4.2""","""мультфильм,фэнтези,семейный""",7355.0,1


In [ ]:
print(f'Описательная статистика:')
df_pl.describe()

Описательная статистика:


statistic,title,puNumber,show_start_date,type,film_studio,production_country,director,producer,age_restriction,refundable_support,nonrefundable_support,budget,financing_source,ratings,genres,total_box_office,show_count
str,str,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,f64
"""count""","""7486""",7485.0,"""7486""","""7486""","""7468""","""7484""","""7477""","""6918""","""7486""","""332""","""332""","""332""","""332""","""6519""","""6510""",3158.0,3158.0
"""null_count""","""0""",1.0,"""0""","""0""","""18""","""2""","""9""","""568""","""0""","""7154""","""7154""","""7154""","""7154""","""967""","""976""",4328.0,4328.0
"""mean""",null,1.3520e8,null,null,null,null,null,null,null,null,null,null,null,null,null,7.6479e7,1.0
"""std""",null,3.8353e7,null,null,null,null,null,null,null,null,null,null,null,null,null,2.4035e8,0.0
"""min""","""""SOS"" над тайгой""",1.811096e6,"""2010-01-11T12:00:00.000Z""","""Анимационный""","""""Дино де Лаурентиис"" (Италия) …","""2019""","""Ёлкин Туйчиев""","""""Фонд Михаила Калатозова""""","""«0+» - для любой зрительской а…","""0.0""","""0.0""","""0.0""","""Министерство культуры""","""1.0""","""аниме,мультфильм""",0.0,1.0
"""25%""",null,1.12025118e8,null,null,null,null,null,null,null,null,null,null,null,null,null,86190.0,1.0
"""50%""",null,1.2101551e8,null,null,null,null,null,null,null,null,null,null,null,null,null,2.330104e6,1.0
"""75%""",null,1.24003314e8,null,null,null,null,null,null,null,null,null,null,null,null,null,2.3983e7,1.0
"""max""","""сНежное шоу""",2.31001111e8,"""2019-12-30T12:00:00.000Z""","""Художественный""","""юФилм, Твинпикс""","""Япония-США-Франция""","""хореография Фредерика Эштона""","""Яэль Фогель, Летиция Гонзалез""","""«6+» - для детей старше 6 лет""","""9000000.0""","""97000000.0""","""980000000.0""","""Фонд кино""","""99%""","""фэнтези,ужасы,триллер""",3.0736e9,1.0


Удалим сразу же фильмы без рейтинга (без целевой переменной), хотя альтернативным решением было бы сделать что-то по типу pseudolabeling или проще (k-means)

Также удалим данные с пропусками в основных категориальных фичах

In [ ]:
for col in ['ratings', 'production_country', 'puNumber', 'film_studio', 'director', 'producer', 'genres']:
    print(f'Deleting null values in {col}...')
    print(f'\tShape before: {df_pl.shape}')
    print(f'\tNull values in {col} before: {df_pl[col].null_count()}')
    df_pl = df_pl.filter(pl.col(col).is_not_null())
    print(f'\tShape after: {df_pl.shape}')
    print(f'\tNull values in {col} after: {df_pl[col].null_count()}\n')

Deleting null values in ratings...
	Shape before: (7486, 17)
	Null values in ratings before: 967
	Shape after: (6519, 17)
	Null values in ratings after: 0

Deleting null values in production_country...
	Shape before: (6519, 17)
	Null values in production_country before: 2
	Shape after: (6517, 17)
	Null values in production_country after: 0

Deleting null values in puNumber...
	Shape before: (6517, 17)
	Null values in puNumber before: 0
	Shape after: (6517, 17)
	Null values in puNumber after: 0

Deleting null values in film_studio...
	Shape before: (6517, 17)
	Null values in film_studio before: 4
	Shape after: (6513, 17)
	Null values in film_studio after: 0

Deleting null values in director...
	Shape before: (6513, 17)
	Null values in director before: 2
	Shape after: (6511, 17)
	Null values in director after: 0

Deleting null values in producer...
	Shape before: (6511, 17)
	Null values in producer before: 477
	Shape after: (6034, 17)
	Null values in producer after: 0

Deleting null valu

У нас есть дубликаты (это точно), но будем оставлять по одной записи на фильм (просуммировов кассовые сборы)

In [ ]:
initial_count = df_pl.height
print(f'Shape before deduplication: {df_pl.shape}')
key_features = ['title']

df_aggregated = df_pl.group_by(key_features).agg(
    pl.sum('total_box_office').alias('total_box_office')
).sort(key_features)

other_columns = [col for col in df_pl.columns if col != 'total_box_office' and col not in key_features]
df_first_occurrence = df_pl.unique(subset=key_features, keep='first').select(key_features + other_columns)

df_unique_key = df_first_occurrence.join(df_aggregated.select(key_features + ['total_box_office']),
                                        on=key_features,
                                        how='left')

duplicates_key = initial_count - df_unique_key.height
print(f'Key-feature duplicates: {duplicates_key}')
print(f'Shape after deduplication: {df_unique_key.shape}')
df_pl = df_unique_key

Shape before deduplication: (6025, 17)
Key-feature duplicates: 691
Shape after deduplication: (5334, 17)


Если после соединия таблиц обнаружили пропуски в `total_box_office` или `show_count`, просто их заменим на 0 (так как ничего не собрали по кинопоказу в РФ в этой прокатной лицензии)

In [ ]:
df_pl = df_pl.with_columns(pl.col("total_box_office", "show_count").fill_null(0))

### 1.1 Общий анализ данных

In [ ]:
print('Схема данных Polars:')
print(*[(k, v) for k, v in df_pl.schema.items()], sep='\n')

Схема данных Polars:
('title', String)
('puNumber', Int64)
('show_start_date', String)
('type', String)
('film_studio', String)
('production_country', String)
('director', String)
('producer', String)
('age_restriction', String)
('refundable_support', String)
('nonrefundable_support', String)
('budget', String)
('financing_source', String)
('ratings', String)
('genres', String)
('show_count', UInt32)
('total_box_office', Float64)


Преобразуем вещественные фичи `refundable_support`, `nonrefundable_support`, `budget`

In [ ]:
columns_to_convert = ['refundable_support', 'nonrefundable_support', 'budget']

df_pl = df_pl.with_columns([
    pl.col(col)
    .str.replace(',', '.')
    .str.replace(r'[^\d.-]', '')  # удаляем все символы кроме цифр, точки и минуса
    .cast(pl.Float64, strict=False)
    .alias(col)
    for col in columns_to_convert
])

Ну и хотелось бы все же преобразовать данные к 1 нормальной форме (чтобы атрибуты не были составными)

In [ ]:
def clean_existing_data(df: pl.DataFrame) -> pl.DataFrame:
    return df.with_columns([
        # Оставляем только первого режиссера
        pl.col("director").str.split(",").list.first().str.split("-").list.first().str.strip_chars(),
        # Оставляем только первого продюсера
        pl.col("producer").str.split(",").list.first().str.split("-").list.first().str.strip_chars(),
        # Оставляем только первый жанр
        pl.col("genres").str.split(",").list.first().str.split("-").list.first().str.strip_chars(),
        # Оставляем только год
        pl.col("show_start_date").str.split("T").list.first().str.slice(0, 4).cast(pl.Int32),
        # Преобразуем рейтинг в один формат (10-бальная шкала)
        pl.when(pl.col("ratings").str.contains("%"))
        .then(pl.col("ratings").str.extract(r"(\d+)").cast(pl.Float64).truediv(10))
        .otherwise(pl.col("ratings").str.replace_all(r"[^\d.]", "").cast(pl.Float64))
    ])

df_pl = clean_existing_data(df_pl)

In [ ]:
df_pl.describe()

statistic,title,puNumber,show_start_date,type,film_studio,production_country,director,producer,age_restriction,refundable_support,nonrefundable_support,budget,financing_source,ratings,genres,show_count,total_box_office
str,str,f64,f64,str,str,str,str,str,str,f64,f64,f64,str,f64,str,f64,f64
"""count""","""5334""",5334.0,5334.0,"""5334""","""5334""","""5334""","""5334""","""5334""","""5334""",312.0,312.0,312.0,"""312""",5334.0,"""5334""",5334.0,5334.0
"""null_count""","""0""",0.0,0.0,"""0""","""0""","""0""","""0""","""0""","""0""",5022.0,5022.0,5022.0,"""5022""",0.0,"""0""",0.0,0.0
"""mean""",null,1.3360e8,2014.652043,null,null,null,null,null,null,1.2224e7,4.9560e7,1.3111e8,null,6.412973,null,0.4985,4.3523e7
"""std""",null,3.6074e7,2.996535,null,null,null,null,null,null,2.5450e7,6.1401e7,1.9351e8,null,1.132815,null,0.500045,1.8588e8
"""min""","""""V"" значит вендетта""",1.1100011e8,2010.0,"""Анимационный""","""""Студия ""Птица Феникс"" Татьяны…","""2019""","""Ёлкин Туйчиев""","""""Фонд Михаила Калатозова""""","""«0+» - для любой зрительской а…",0.0,0.0,0.0,"""Министерство культуры""",1.0,"""аниме""",0.0,0.0
"""25%""",null,1.21001013e8,2012.0,null,null,null,null,null,null,0.0,2.5e7,4.2e7,null,5.8,null,0.0,0.0
"""50%""",null,1.21016312e8,2015.0,null,null,null,null,null,null,0.0,3e7,7.0305e7,null,6.6,null,0.0,465.0
"""75%""",null,1.24000219e8,2017.0,null,null,null,null,null,null,1.5e7,4.15e7,1.5e8,null,7.2,null,1.0,3.677875e6
"""max""","""Ярость / Fury""",2.31001012e8,2019.0,"""Художественный""","""юФилм, Твинпикс""","""Япония-США-Франция""","""Яш Чопра""","""Яэль Фогель""","""«6+» - для детей старше 6 лет""",1.8e8,4e8,2.3051e9,"""Фонд кино""",9.9,"""фэнтези""",1.0,3.0736e9


Картина чрезвычайно печальная (по информации о финансировании 5700 кортежей с пропусками). Стоит ли в целом сохранять вещественные признаки (и в целом информаицию по финансированию) для предсказания рейтинга, если таких записей только 300 кортежей?

В теории, опять же, как точка роста: попытаться обогатить датасет дополнительными данными о бюджете

А пока просто удалим эти признаки

In [ ]:
columns_with_nans = [col for col in df_pl.columns if df_pl[col].null_count() > 0]
df_pl = df_pl.drop(columns_with_nans)
enriched_df = df_pl

 Показываем уникальные значения для (потенциально) категориальных признаков

In [ ]:
from prettytable import PrettyTable

threshold_nunique = 30
uniques_tb = PrettyTable()
uniques_tb.field_names = ['feature', 'n_unique', f'Values (if quantity less than {threshold_nunique})']
category_like_columns = []

for col in enriched_df.columns:
    unique_count = enriched_df[col].n_unique()
    if unique_count < threshold_nunique:
        category_like_columns.append(col)
        unique_values = enriched_df[col].drop_nulls().unique().sort().to_list()
        values_str = ', '.join(map(str, unique_values))
    else:
        values_str = '-'

    uniques_tb.add_row([col, unique_count, values_str])

print(uniques_tb)

+--------------------+----------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|      feature       | n_unique |                                                                                                                           Values (if quantity less than 30)                                                                                                                            |
+--------------------+----------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|       title        |   5334   |                      

Теперь выведем таблицы для каждого (потенциально) категориального признака и посмотрим распределение сэмплов по каждому из значений. Дальше при обработке я буду руководствоваться похожими агрегатами, чтобы удалять из сэмплов нерепрезентативные классы.

In [ ]:
from prettytable import PrettyTable

def print_unique_values_table(df, category):
    print(f'\n{"="*50}')
    print(f'Unique values in {category}:')
    print(f'\n{"="*50}')
    category_counts = df.group_by(category).agg(pl.count().alias('count')).sort('count', descending=True)
    table = PrettyTable()
    table.field_names = [category, 'count', 'percent (%)']

    total_count = category_counts['count'].sum()
    for row in category_counts.iter_rows(named=True):
        value = row[category]
        count = row['count']
        percentage = (count / total_count) * 100
        table.add_row([value, count, f'{percentage:.1f}%'])

    print(table)

In [ ]:
df_cleared = df_pl.clone()

In [ ]:
for category in category_like_columns:
    print_unique_values_table(df_cleared, category)


Unique values in show_start_date:

+-----------------+-------+-------------+
| show_start_date | count | percent (%) |
+-----------------+-------+-------------+
|       2010      |  670  |    12.6%    |
|       2019      |  665  |    12.5%    |
|       2018      |  635  |    11.9%    |
|       2016      |  618  |    11.6%    |
|       2015      |  514  |     9.6%    |
|       2014      |  487  |     9.1%    |
|       2013      |  469  |     8.8%    |
|       2012      |  458  |     8.6%    |
|       2011      |  421  |     7.9%    |
|       2017      |  397  |     7.4%    |
+-----------------+-------+-------------+

Unique values in type:

+----------------------------+-------+-------------+
|            type            | count | percent (%) |
+----------------------------+-------+-------------+
|       Художественный       |  4475 |    83.9%    |
|        Анимационный        |  495  |     9.3%    |
|           Прочие           |  180  |     3.4%    |
|       Документальный       |  1

Удостоверимся, что пропущенных значений больше нет

In [ ]:
def analyze_missing_values_by_column(df):
    total_rows = df.height

    missing_analysis = df.select([
        pl.col(col).null_count().alias(f'{col}_missing') for col in df.columns
    ]).transpose(include_header=True, header_name='column', column_names=['missing_count'])

    missing_df = missing_analysis.with_columns([
        (pl.col('missing_count') / total_rows * 100).alias('missing_percent')
    ]).sort('missing_percent', descending=True)

    missing_with_nulls = missing_df.filter(pl.col('missing_count') > 0)
    if len(missing_with_nulls) == 0:
        return "No more missing values!"

    print(missing_with_nulls)

    missing_data = {
        'column': missing_with_nulls['column'].to_list(),
        'missing_percent': missing_with_nulls['missing_percent'].to_list()
    }
    fig = px.bar(
        missing_data,
        x='column',
        y='missing_percent',
        title='Missing values percent by column',
        labels={'missing_percent': 'Missing values percent', 'column': 'columns'},
        color='missing_percent',
        color_continuous_scale='Reds'
    )
    fig.update_layout(
        xaxis_tickangle=-45,
        height=500,
        showlegend=False
    )
    fig.show()

analyze_missing_values_by_column(df_cleared)

'No more missing values!'

### 1.2 Обработка аномальных значений и выбросов


In [ ]:
numeric_features = ['ratings', 'total_box_office']
print('Numeric features:', numeric_features)

Numeric features: ['ratings', 'total_box_office']


Взглянем также на распределение каждого признака и соответсвующее ему нормальное.

Применим для каждого вещественного признака квантильное преобразование, чтобы гарантированно получить нормальные распределения.

In [ ]:
from sklearn.preprocessing import QuantileTransformer

def quantile_transform_visualize(df, columns):
    df_transformed = df.clone()

    fig = make_subplots(rows=len(columns), cols=2,
        subplot_titles=[f'{col} ({suffix})' for col in columns
                       for suffix in ['before', 'after']],
        vertical_spacing=0.05)
    for i, col in enumerate(columns):
        original_data = df[col].to_numpy()
        transformer = QuantileTransformer(output_distribution='normal', random_state=42)
        transformed_data = transformer.fit_transform(original_data.reshape(-1, 1)).flatten()

        df_transformed = df_transformed.with_columns(
            pl.Series(col, transformed_data)
        )

        fig.add_trace(go.Histogram(x=original_data, showlegend=False), row=i+1, col=1)
        fig.add_trace(go.Histogram(x=transformed_data, showlegend=False), row=i+1, col=2)

    fig.update_layout(height=300 * len(columns), title_text="Quantile Transformation")
    fig.show()

    return df_transformed

df_processed = quantile_transform_visualize(df_cleared, numeric_features)

Отбор нужных нам колонок:

In [ ]:
target_column = 'ratings'

index_features = ['puNumber', 'title']
useless_features = ['film_studio', 'production_country', 'director', 'producer']
non_numeric_features = category_like_columns + index_features + useless_features

categoric_features_in_use = category_like_columns
numeric_features_in_use = [col for col in df_processed.columns if col not in non_numeric_features]

In [ ]:
print('Numeric features:', numeric_features_in_use)
print('Categoric features:', categoric_features_in_use)
print('Target feature:', target_column)
FEATURES_IN_USE = numeric_features_in_use + categoric_features_in_use

Numeric features: ['ratings', 'total_box_office']
Categoric features: ['show_start_date', 'type', 'age_restriction', 'genres', 'show_count']
Target feature: ratings


### 1.3. Кодирование и скалирование

Анализ распределений до этого момента был достаточно сумбурным (да и сейчас ничего не поменяется), но я специально вел отдельно учет категориальных признаков (чтобы применить к ним one-hot encoding), отдельно логарифмированных признаков (чтобы применить к ним скейлинг).

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

df_normed = df_processed.select(FEATURES_IN_USE).to_pandas()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features_in_use),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categoric_features_in_use)
    ]
)

transformed_data = preprocessor.fit_transform(df_normed)
categorical_names = preprocessor.named_transformers_['cat'].get_feature_names_out(categoric_features_in_use)
feature_names = np.concatenate([numeric_features_in_use, categorical_names])

df_normed = pd.DataFrame(transformed_data, columns=feature_names)
df_normed

,ratings,total_box_office,show_start_date_2011,show_start_date_2012,show_start_date_2013,show_start_date_2014,show_start_date_2015,show_start_date_2016,show_start_date_2017,show_start_date_2018,...,genres_приключения,genres_реальное ТВ,genres_семейный,genres_спорт,genres_триллер,genres_ужасы,genres_фантастика,genres_фильм,genres_фэнтези,show_count_1
0,0.884225,-1.021709,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,-1.549013,-1.021709,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,-1.308016,1.225109,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,0.884225,1.464484,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
4,-2.713049,0.770790,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5329,-0.207535,-1.021709,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
5330,-0.493608,0.736768,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
5331,-0.669804,0.689372,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
5332,0.691183,-1.021709,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 2. Построение baseline-модели нейронной сети

1. Создайте класс для архитектуры нейронной сети (используйте фреймворк на выбор — Keras, PyTorch, TensorFlow)

Самостоятельно выберите:
* Количество скрытых слоёв
* Количество нейронов в каждом слое
* Функции активации для скрытых и выходного слоёв
* Реализуйте обучение модели: напишите функцию обучения и выберите метрику для оценки

In [ ]:
features = [col for col in df_normed.columns if col != target_column]

X, y = df_normed[features], df_normed[target_column]

In [ ]:
from sklearn.model_selection import train_test_split

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.1, random_state=seed)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.22, random_state=seed)

print(f'Train subset: {X_train.shape[0]} ({X_train.shape[0]/X.shape[0]:.2%})')
print(f'Val subset: {X_val.shape[0]} ({X_val.shape[0]/X.shape[0]:.2%})')
print(f'Test subset: {X_test.shape[0]} ({X_test.shape[0]/X.shape[0]:.2%})')

Train subset: 3744 (70.19%)
Val subset: 1056 (19.80%)
Test subset: 534 (10.01%)


In [ ]:
import torch
from torch.utils.data import Dataset

class MovieDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X.values if hasattr(X, 'values') else X)
        self.y = torch.FloatTensor(y.values if hasattr(y, 'values') else y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

* BatchNorm после Linear: стабилизирует распределение активаций, ускоряет обучение

* ReLU после BatchNorm: стандартная практика - нелинейность после нормализации

* Dropout в конце: регуляризация для предотвращения переобучения

In [ ]:
import torch.nn as nn

activation_funcs = {
    'relu': nn.ReLU(),
    'leaky_relu': nn.LeakyReLU(0.1),
    'elu': nn.ELU(),
    'tanh': nn.Tanh()
}

class MovieRatingPredictor(nn.Module):
    def __init__(self, input_size, hidden_layers, dropout_rate=0.3,
                 use_batchnorm=True, activation_fn='relu'):
        super().__init__()

        layers = []
        prev_size = input_size
        for hidden_size in hidden_layers:
            layers.append(nn.Linear(prev_size, hidden_size))
            if use_batchnorm:
                layers.append(nn.BatchNorm1d(hidden_size))
            layers.append(activation_funcs[activation_fn])
            layers.append(nn.Dropout(dropout_rate))
            prev_size = hidden_size
        layers.append(nn.Linear(prev_size, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x).squeeze()

In [ ]:
from torch.optim.swa_utils import AveragedModel, SWALR

def train_model(model, train_loader, val_loader, criterion, optimizer,
                num_epochs=1000, patience=30, swa_start=800, swa_lr=1e-5):
    train_losses, val_losses = [], []
    best_val_loss = float('inf')
    patience_counter = 0
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,
                                                           mode='min',
                                                           factor=0.5,
                                                           patience=patience)
    swa_model = AveragedModel(model)
    swa_scheduler = SWALR(optimizer, swa_lr=swa_lr)
    swa_started = False

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()

        if epoch >= swa_start:
            if not swa_started:
                swa_started = True
            swa_model.update_parameters(model)
            swa_scheduler.step()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                val_loss += loss.item()

        train_loss /= len(train_loader)
        val_loss /= len(val_loader)
        scheduler.step(val_loss)
        train_losses.append(train_loss)
        val_losses.append(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), 'best_model.pth')
        else:
            patience_counter += 1

        if patience_counter >= patience:
            break

    if swa_started:
        torch.optim.swa_utils.update_bn(train_loader, swa_model)
        torch.save(swa_model.module.state_dict(), 'best_swa_model.pth')
        model.load_state_dict(torch.load('best_swa_model.pth'))
    else:
        model.load_state_dict(torch.load('best_model.pth'))

    return train_losses, val_losses

In [ ]:
def evaluate_model(model, test_loader, criterion):
    model.eval()
    test_loss = 0.0
    predictions, actuals = [], []

    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            test_loss += loss.item()
            predictions.extend(outputs.cpu().numpy())
            actuals.extend(batch_y.cpu().numpy())

    test_loss /= len(test_loader)
    predictions = np.array(predictions)
    actuals = np.array(actuals)
    mae = np.mean(np.abs(predictions - actuals))
    rmse = np.sqrt(np.mean((predictions - actuals)**2))
    r2 = 1 - np.sum((actuals - predictions)**2) / np.sum((actuals - np.mean(actuals))**2)
    return test_loss, mae, rmse, r2, predictions, actuals

In [ ]:
# @title
def create_interactive_plots(train_losses, val_losses, predictions, actuals, metrics):
    fig = make_subplots(rows=2, cols=2,
        subplot_titles=('График обучения: Train vs Validation Loss',
                        'Предсказания vs Фактические значения',
                        'Распределение ошибок',
                        'Метрики качества модели'))

    epochs = list(range(1, len(train_losses) + 1))
    fig.add_trace(go.Scatter(x=epochs, y=train_losses,
                            name='Train Loss', mode='lines'), row=1, col=1)
    fig.add_trace(go.Scatter(x=epochs, y=val_losses,
                            name='Validation Loss', mode='lines'), row=1, col=1)
    fig.update_xaxes(title_text="Epochs", row=1, col=1)
    fig.update_yaxes(title_text="Loss (MSE)", row=1, col=1)

    fig.add_trace(go.Scatter(x=actuals, y=predictions, mode='markers',
                            name='Pred',
                            marker=dict(opacity=0.6)), row=1, col=2)

    fig.update_xaxes(title_text="Actual", row=1, col=2)
    fig.update_yaxes(title_text="Pred", row=1, col=2)

    errors = predictions - actuals
    fig.add_trace(go.Histogram(x=errors,
                              nbinsx=50, name='Error distribution', opacity=0.7),
                 row=2, col=1)
    fig.update_xaxes(title_text="Predicted - Actual", row=2, col=1)
    fig.update_yaxes(title_text="Freq", row=2, col=1)

    metrics_names = ['MSE', 'MAE', 'RMSE', 'R2']
    metrics_values = [metrics['mse'], metrics['mae'],
                      metrics['rmse'], metrics['r2']]

    fig.add_trace(go.Bar(x=metrics_names, y=metrics_values,
                        text=[f'{val:.4f}' for val in metrics_values],
                        textposition='auto'), row=2, col=2)
    fig.update_xaxes(title_text="Metrics", row=2, col=2)
    fig.update_yaxes(title_text="Values", row=2, col=2)

    fig.update_layout(height=800, showlegend=True)
    fig.show()

In [ ]:
from torch.utils.data import DataLoader

train_dataset = MovieDataset(X_train, y_train)
val_dataset = MovieDataset(X_val, y_val)
test_dataset = MovieDataset(X_test, y_test)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

input_size = X_train.shape[1]
model = MovieRatingPredictor(input_size=input_size,
                             hidden_layers=[512, 256, 128, 64],
                             dropout_rate=0.5)

criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

train_losses, val_losses = train_model(model,
                                       train_loader, val_loader,
                                       criterion, optimizer,
                                       num_epochs=500, patience=50)

А теперь запустим на тестовой подвыборке

In [ ]:
test_loss, mae, rmse, r2, predictions, actuals = evaluate_model(model, test_loader, criterion)

metrics = {'mse': test_loss, 'mae': mae, 'rmse': rmse, 'r2': r2}
print(metrics)
create_interactive_plots(train_losses, val_losses, predictions, actuals, metrics)

{'mse': 0.7798045628211078, 'mae': np.float32(0.6887928), 'rmse': np.float32(0.88223845), 'r2': np.float32(0.15684253)}


Низкая объясняющая способность: метрика r2 имеет катастрофически низкие значения, что говорит о неспособности модели уловить закономерности в данных.

Признак переобучения: loss на тренировочной выборке устойчиво уменьшается, однако на валидационной выборке не показывает признаков сходимости, её поведение напоминает случайный процесс.

Предварительная гипотеза: Основная проблема лежит не в области архитектуры модели или алгоритма обучения, а в самих данных

## Доработка модели

Реализуйте подбор параметров (grid search) для следующих гиперпараметров:
* Функция активации.
* Dropout (наличие и значение).
* Batch Normalization (наличие).
* Размер батча.
* Архитектуру (количество слоёв и нейронов) оставьте как в Baseline.


In [ ]:
import itertools

def grid_search(param_grid, train_dataset, val_dataset, test_dataset,
                input_size, hidden_layers, num_epochs=500, patience=20):
    results = []
    keys = list(param_grid.keys())
    values = [param_grid[key] for key in keys]

    for combination in itertools.product(*values):
        params = dict(zip(keys, combination))
        train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=params['batch_size'], shuffle=True)
        val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=params['batch_size'])
        test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=params['batch_size'])

        model = MovieRatingPredictor(input_size=input_size,
                                     hidden_layers=hidden_layers,
                                     dropout_rate=params['dropout_rate'],
                                     use_batchnorm=params['use_batchnorm'],
                                     activation_fn=params['activation_fn'])

        criterion = nn.MSELoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

        train_losses, val_losses = train_model(model, train_loader, val_loader,
                                               criterion, optimizer, num_epochs=num_epochs, patience=patience)

        test_loss, mae, rmse, r2, predictions, actuals = evaluate_model(model, test_loader, criterion)
        result = params.copy()
        result.update({'test_loss': test_loss, 'mae': mae, 'rmse': rmse, 'r2': r2})
        results.append(result)

        print(f'Params: {params}, RMSE: {rmse:.4f}')

    return results

In [ ]:
param_grid = {
    'activation_fn': ['relu', 'leaky_relu', 'elu', 'tanh'],
    'dropout_rate': [0.0, 0.1, 0.2, 0.5],
    'use_batchnorm': [True, False],
    'batch_size': [32, 64, 128, 256, 512]
}

grid_search_results = grid_search(param_grid=param_grid,
                                  train_dataset=train_dataset,
                                  val_dataset=val_dataset,
                                  test_dataset=test_dataset,
                                  input_size=X_train.shape[1],
                                  hidden_layers=[512, 256, 128, 64],
                                  num_epochs=500, patience=50)

Params: {'activation_fn': 'relu', 'dropout_rate': 0.0, 'use_batchnorm': True, 'batch_size': 32}, RMSE: 0.8745
Params: {'activation_fn': 'relu', 'dropout_rate': 0.0, 'use_batchnorm': True, 'batch_size': 64}, RMSE: 0.8747
Params: {'activation_fn': 'relu', 'dropout_rate': 0.0, 'use_batchnorm': True, 'batch_size': 128}, RMSE: 0.8839
Params: {'activation_fn': 'relu', 'dropout_rate': 0.0, 'use_batchnorm': True, 'batch_size': 256}, RMSE: 0.8817
Params: {'activation_fn': 'relu', 'dropout_rate': 0.0, 'use_batchnorm': True, 'batch_size': 512}, RMSE: 0.8921
Params: {'activation_fn': 'relu', 'dropout_rate': 0.0, 'use_batchnorm': False, 'batch_size': 32}, RMSE: 0.8728
Params: {'activation_fn': 'relu', 'dropout_rate': 0.0, 'use_batchnorm': False, 'batch_size': 64}, RMSE: 0.8881
Params: {'activation_fn': 'relu', 'dropout_rate': 0.0, 'use_batchnorm': False, 'batch_size': 128}, RMSE: 0.8894
Params: {'activation_fn': 'relu', 'dropout_rate': 0.0, 'use_batchnorm': False, 'batch_size': 256}, RMSE: 0.8699
P

Визуализируйте зависимость метрики RMSE от значений подбираемых параметров.



In [ ]:
import json

def visualize_grid_search_results(grid_search_results):
    df = pd.DataFrame(grid_search_results)
    df['use_batchnorm'] = df['use_batchnorm'].astype(str)
    df['batch_size'] = df['batch_size'].astype(str)

    fig = px.parallel_categories(df, dimensions=['activation_fn', 'dropout_rate',
                                                'use_batchnorm', 'batch_size'],
                                color='rmse', color_continuous_scale='inferno')
    fig.update_layout(coloraxis_colorbar=dict(title="RMSE"))
    fig.show()

    best_result = df.loc[df['rmse'].idxmin()]
    return best_result

best_config = visualize_grid_search_results(grid_search_results)
print(f"Best configuration: {json.dumps(best_config.to_dict(), indent=4)}")

Best configuration: {
    "activation_fn": "elu",
    "dropout_rate": 0.5,
    "use_batchnorm": "False",
    "batch_size": "256",
    "test_loss": 0.7825613220532736,
    "mae": 0.6783846020698547,
    "rmse": 0.8595691323280334,
    "r2": 0.19961601495742798
}


1. `"activation_fn": "elu"` (Exponential Linear Unit) лучше ReLU тем, что не "умирает" при отрицательных входах

2. `"dropout_rate": 0.5` (очень высокий!) - агрессивная регуляризация (модель вынуждена учиться более robust-представлениям)

3. `"use_batchnorm": "False"`
отсутствие BatchNorm при таком высоком dropout интересно, данные уже предобработаны

## Выводы

Формально условия лабораторной выполнены :)

Попытка натянуть сову на глобус редко приводит к хорошему результату. Если данные сами по себе не содержат полезных признаков, содержат много категориальных переменных, которые не должны сильно влиять на целевую переменную, то и ждать от модели, будь то регрессионная статистическая или нейросетевая, выдающихся метрик не стоит.

Полагаю, даже простой ансамбль мог бы показать сравнимые результаты. Однако в рамках лабораторной работы мы решили использовать существующие инструменты в рамках ознакомления.

В погоне за хайпом для этого датасета оптимальным решением будет обогащение данных с помощью MAS (если есть поддержка анализа браузерной выдачи). Затем мы можем применить zero-shot предиктор на основе БЯМ. Можно объединить все имеющиеся атрибуты, которые трудно использовать (например, информацию о режиссере, продюсере, кинокомпании, стране производства и модели финансирования), чтобы создать текстовый промпт с описанием фильма и вопросом: «А если бы вы ставили оценку этому фильму на Кинопоиске...».

Было бы интересно попробовать посмотреть на результаты работы такого подхода в сравнении с представленным в этой лабе.


Вот и все, конец!

**Лабораторная работа выполнена в рамках курса "Машинное обучение" ИТМО, 2025**